In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

from huggingface_hub import hf_hub_download
import keras
import joblib
import json

repo_id = "jengyang/lstm-stock-prediction-model"

model_path = hf_hub_download(
    repo_id=repo_id,
    filename="stage2_universal_lstm_20250705_170829.keras"
)

scaler_path = hf_hub_download(
    repo_id=repo_id,
    filename="stage2_scalers_20250705_170829.pkl"
)

metadata_path = hf_hub_download(
    repo_id=repo_id,
    filename="stage2_metadata_20250705_170829.json"
)

model = keras.saving.load_model(model_path, compile=False)

scalers = joblib.load(scaler_path)

with open(metadata_path, "r") as f:
    metadata = json.load(f)

model.summary()
print(metadata)

print("Scaler type:", type(scalers))

if isinstance(scalers, dict):
    print("Scaler keys:", scalers.keys())
else:
    print(scalers)

/Users/sahas/Documents/StockRecommendationApp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/sahas/Documents/StockRecommendationApp/.venv/lib/python3.13/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm1 (LSTM)                    │ (None, 60, 50)         │        11,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout1 (Dropout)              │ (None, 60, 50)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm2 (LSTM)                    │ (None, 50)             │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout2 (Dropout)              │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,651 (123.64 KB)

 Trainable params: 31,651 (123.64 KB)

 Non-trainable params: 0 (0.00 B)

{'model_name': 'Stage 2 Simple Universal LSTM', 'stocks': ['AAPL', 'AMZN', 'AVGO', 'BRK.B', 'COST', 'GOOG', 'JNJ', 'JPM', 'LLY', 'MA', 'META', 'MSFT', 'NFLX', 'NVDA', 'ORCL', 'PG', 'TSLA', 'V', 'WMT', 'XOM'], 'features': ['Open', 'High', 'Low', 'Close', 'Volume', 'sentiment_10d_avg'], 'target': 'Close', 'time_steps': 60, 'batch_size': 16, 'epochs_trained': 88, 'final_val_loss': 0.0008096768870018423, 'rmse': 26.850322364411017, 'mae': 12.60320354323313, 'r2': 0.9902821175992801, 'mape': 2.679250893651326, 'created_at': '20250705_170829', 'architecture': {'lstm1_units': 50, 'lstm2_units': 50, 'dropout_rate': 0.2, 'dense_units': 1}}
Scaler type: <class 'dict'>
Scaler keys: dict_keys(['feature_scaler', 'target_scaler'])


In [2]:
print(type(scalers))

if isinstance(scalers, dict):
    print(scalers.keys())
else:
    print(scalers)

<class 'dict'>
dict_keys(['feature_scaler', 'target_scaler'])


In [3]:
import numpy as np

X_dummy = np.random.rand(1, 60, 6)
y_pred = model.predict(X_dummy)

print(y_pred)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
[[0.72066295]]


In [7]:
import numpy as np
import pandas as pd
import tensorflow as tf
import yfinance as yf

ticker = "AAPL"

# 1. Fetch data safely
# 'auto_adjust=True' ensures OHLC values match historical corporate stock splits
df = yf.download(ticker, period="6mo", interval="1d", auto_adjust=True)

# 2. Add '.copy()' to eliminate potential SettingWithCopy warnings
df = df[["Open", "High", "Low", "Close", "Volume"]].copy()

# 3. Inject LLM Scores (Using 0.0 as a neutral placeholder for testing)
# In production, replace this with a dynamic array of your daily LLM news scores (-1 to +1)
df["Sentiment"] = 0.0

# 4. Extract trailing sequence window
latest_60 = df.tail(60).values

# 5. Fail-Safe: Verify matrix constraints match the LSTM input shape rules
if latest_60.shape[0] != 60:
    raise ValueError(
        f"Data footprint mismatch. Got {latest_60.shape[0]} rows, but LSTM requires exactly 60 rows."
    )

# 6. Apply preprocessing scalers
latest_60_scaled = scalers["feature_scaler"].transform(latest_60)

# 7. Reshape to 3D Tensor format: (Batch Size, Timesteps, Features)
X = latest_60_scaled.reshape(1, 60, 6)

# 8. Inference Execution
raw_prediction = model.predict(X, verbose=0)  # verbose=0 suppresses console log clutter

# 9. Invert target back into currency space
predicted_price = scalers["target_scaler"].inverse_transform(raw_prediction)[0][0]

print(f"Predicted next close price for {ticker}: ${predicted_price:.2f}")

[*********************100%***********************]  1 of 1 completed

Predicted next close price for AAPL: $309.78
